# 00download: daily ridership from the Seoul Open API

Downloads daily boardings and alightings by line and station (dataset `CardSubwayStatsNew`, the same data as the CARD_SUBWAY files) for every day from the start month to the end month.

- One request per day (about 600 rows a day, the API allows 1,000 per request).
- Saved as **one CSV per month** in `data/raw/api/`. A month is written only when all its days are in, so a stopped run loses at most one month of work.
- **Resumable:** months that already have a file are skipped. Just run the notebook again.
- The API key is read from `seoul_api_key.txt` and is never printed or saved anywhere else.

## Section 0: settings

In [ ]:
from pathlib import Path

KEY_FILE = Path("seoul_api_key.txt")                  # <- text file holding only your API key (never upload it)
BASE_URL = "http://openapi.seoul.go.kr:8088"          # <- Seoul Open API address
SERVICE = "CardSubwayStatsNew"                        # <- dataset name in the API
OUT_DIR = Path("data/raw/api")                        # <- where the monthly CSV files go

START_MONTH = "2026-03"                               # <- first month to download
END_MONTH = "2026-09"                                 # <- last month to download (use complete months only)

PAUSE_S = 0.05                                        # <- pause between requests, in seconds (be polite to the server)
RETRIES = 5                                           # <- tries per request before giving up on that day
TIMEOUT_S = 30                                        # <- seconds to wait for one answer

## Section 1: tools

In [2]:
import json
import time
import urllib.request
import urllib.error

import pandas as pd

OUT_DIR.mkdir(parents=True, exist_ok=True)
API_KEY = KEY_FILE.read_text(encoding="utf-8-sig").strip()
print(f"key loaded ({len(API_KEY)} characters)")

# API field names and the column names used in the downloaded CARD_SUBWAY files
COLS = {
    "USE_YMD": "사용일자",
    "SBWY_ROUT_LN_NM": "노선명",
    "SBWY_STNS_NM": "역명",
    "GTON_TNOPE": "승차총승객수",
    "GTOFF_TNOPE": "하차총승객수",
    "REG_YMD": "등록일자",
}


class StopDownload(Exception):
    """Raised when the API says to stop (for example a bad key or a daily limit)."""


def fetch_day(day, page_size=1000):
    """Return all rows for one day (a list of dicts). An empty list means the API has no data for that day."""
    rows, start = [], 1
    while True:
        url = f"{BASE_URL}/{API_KEY}/json/{SERVICE}/{start}/{start + page_size - 1}/{day}"
        for attempt in range(1, RETRIES + 1):
            try:
                with urllib.request.urlopen(url, timeout=TIMEOUT_S) as r:
                    data = json.loads(r.read().decode("utf-8"))
                break
            except (urllib.error.URLError, TimeoutError, json.JSONDecodeError, ConnectionError) as e:
                if attempt == RETRIES:
                    raise
                time.sleep(2 ** attempt)  # wait 2, 4, 8, 16 seconds before trying again

        block = data.get(SERVICE)
        if block is None:  # error answers come without the dataset block
            result = data.get("RESULT", {})
            code_ = result.get("CODE", "?")
            if code_ == "INFO-200":  # no data for this day
                return []
            raise StopDownload(f"{code_}: {result.get('MESSAGE', data)}")

        rows += block.get("row", [])
        total = int(block.get("list_total_count", 0))
        if start + page_size - 1 >= total:
            return rows
        start += page_size


def months_between(first, last):
    return list(pd.period_range(first, last, freq="M"))


def show_progress(done, total, t0, note=""):
    """One-line progress bar: done/total, time so far, time left."""
    elapsed = time.time() - t0
    rate = done / elapsed if elapsed > 0 else 0
    left = (total - done) / rate if rate > 0 else 0
    width = 30
    filled = int(width * done / total) if total else width
    bar = "#" * filled + "-" * (width - filled)
    fmt = lambda s: f"{int(s // 3600)}:{int(s % 3600 // 60):02d}:{int(s % 60):02d}"
    print(f"\r[{bar}] {done}/{total} days | {fmt(elapsed)} so far | {fmt(left)} left {note}   ", end="", flush=True)

key loaded (30 characters)


## Section 2: download

Run this cell. It can be stopped at any time (the square stop button) and restarted later: finished months are kept and skipped.

In [3]:
months = months_between(START_MONTH, END_MONTH)
todo = [m for m in months if not (OUT_DIR / f"CARD_SUBWAY_API_{m.strftime('%Y%m')}.csv").exists()]
days_total = sum(m.days_in_month for m in todo)
print(f"months in range: {len(months)}, already done: {len(months) - len(todo)}, to download: {len(todo)} ({days_total} days)")

empty_days = []
done, t0 = 0, time.time()
try:
    for m in todo:
        month_rows = []
        for d in pd.date_range(m.start_time, m.end_time, freq="D"):
            day = d.strftime("%Y%m%d")
            rows = fetch_day(day)
            if not rows:
                empty_days.append(day)
            month_rows += rows
            done += 1
            show_progress(done, days_total, t0, note=f"| {m}")
            time.sleep(PAUSE_S)
        if not month_rows:  # nothing at all: do not save, so the month is tried again next run
            print(f"\n{m}: no data for any day, not saved")
            continue
        out = OUT_DIR / f"CARD_SUBWAY_API_{m.strftime('%Y%m')}.csv"
        tmp = out.with_suffix(".tmp")
        pd.DataFrame(month_rows, columns=list(COLS)).rename(columns=COLS).to_csv(tmp, index=False, encoding="utf-8-sig")
        tmp.replace(out)  # the file only appears once the month is complete
    print("\nfinished")
except StopDownload as e:
    print(f"\nstopped by the API: {e}. Finished months are saved; run this cell again later.")
except KeyboardInterrupt:
    print("\nstopped by you. Finished months are saved; run this cell again to continue.")

if empty_days:
    print(f"days with no data ({len(empty_days)}): {empty_days[:20]}{' ...' if len(empty_days) > 20 else ''}")

months in range: 141, already done: 0, to download: 141 (4291 days)
[------------------------------] 31/4291 days | 0:00:03 so far | 0:08:35 left | 2015-01   
2015-01: no data for any day, not saved
[------------------------------] 59/4291 days | 0:00:07 so far | 0:08:25 left | 2015-02   
2015-02: no data for any day, not saved
[------------------------------] 90/4291 days | 0:00:10 so far | 0:08:21 left | 2015-03   
2015-03: no data for any day, not saved
[------------------------------] 120/4291 days | 0:00:14 so far | 0:08:19 left | 2015-04   
2015-04: no data for any day, not saved
[#-----------------------------] 151/4291 days | 0:00:18 so far | 0:08:20 left | 2015-05   
2015-05: no data for any day, not saved
[#-----------------------------] 181/4291 days | 0:00:21 so far | 0:08:14 left | 2015-06   
2015-06: no data for any day, not saved
[#-----------------------------] 212/4291 days | 0:00:25 so far | 0:08:12 left | 2015-07   
2015-07: no data for any day, not saved
[#---------

## Section 3: check the result

Counts the monthly files and, for the months you also downloaded by hand (2019, 2021, September 2026), compares the daily totals with those files.

In [4]:
files = sorted(OUT_DIR.glob("CARD_SUBWAY_API_*.csv"))
print(f"monthly files: {len(files)} of {len(months)}")

api = pd.concat([pd.read_csv(f, dtype=str, encoding="utf-8-sig") for f in files], ignore_index=True)
for c in ["승차총승객수", "하차총승객수"]:
    api[c] = pd.to_numeric(api[c], errors="coerce")
api["사용일자"] = pd.to_datetime(api["사용일자"], format="%Y%m%d")
print(f"rows: {len(api):,}, days: {api['사용일자'].nunique()}, from {api['사용일자'].min():%Y-%m-%d} to {api['사용일자'].max():%Y-%m-%d}")

# rows per year and average daily boardings (millions)
by_year = api.groupby(api["사용일자"].dt.year).agg(
    rows=("역명", "size"), days=("사용일자", "nunique"), boardings=("승차총승객수", "sum"))
by_year["per_day_million"] = (by_year["boardings"] / by_year["days"] / 1e6).round(2)
by_year.drop(columns="boardings")

monthly files: 8 of 141
rows: 132,669, days: 215, from 2025-08-19 to 2026-09-30


,rows,days,per_day_million
사용일자,,,
2025,616,1,7.52
2026,132053,214,7.27


In [5]:
# compare with the files downloaded by hand (daily total boardings)
import csv, io, re

def read_manual(path):
    raw = path.read_bytes()
    for enc in ["utf-8-sig", "cp949"]:
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    text = re.sub(r'(\d")("\d{8}",")', r"\1\n\2", text)
    rows = [r[:6] for r in csv.reader(io.StringIO(text.lstrip("\ufeff")))][1:]
    df = pd.DataFrame([r for r in rows if r], columns=list(COLS.values()))
    df["사용일자"] = pd.to_datetime(df["사용일자"], format="%Y%m%d")
    df["승차총승객수"] = pd.to_numeric(df["승차총승객수"], errors="coerce")
    return df

manual = pd.concat([read_manual(f) for f in sorted(Path("data/raw/ridership").glob("*.csv"))])
a = api.groupby("사용일자")["승차총승객수"].sum()
b = manual.groupby("사용일자")["승차총승객수"].sum()
both = pd.concat([a.rename("api"), b.rename("manual")], axis=1, join="inner")
both["diff"] = both["api"] - both["manual"]
print(f"days compared: {len(both)}, days with a difference: {(both['diff'] != 0).sum()}")
both[both["diff"] != 0].head(10)

days compared: 30, days with a difference: 0


,api,manual,diff
사용일자,,,


___
**END**